# E6: Is global cross-modal structure there when both sides have a *global* representation?

**Question (Mayug's hypothesis).** The Aristotelian paper finds that global alignment (CKA) disappears after calibration while local alignment (mutual k-NN) survives. They, like Huh et al., measured **mean-pooled hidden states of decoder LLMs**. Hypothesis: global structure is shared once the language side has a global sentence representation, e.g. an encoder, or a decoder turned into one.

**Design (2 × 2 + pooling).** WIT-1024 (Huh et al.), the same pipeline and engine as the E2 audit (v3 engine, all layer pairs, permutation null on the max).
- **Language:** Qwen3 decoders (0.6B / 4B / 8B **Base**) vs the Qwen3-Embedding models of the same sizes (decoder architecture turned into an embedding model), plus mpnet (a classic encoder).
  - Readouts: `mean` (E2 / Huh protocol), `last` token, and for decoders `prompt` (PromptEOL, a training-free decoder→encoder readout). For mpnet: `mean` (its native pooling) and `cls`.
- **Vision:** MAE (masked reconstruction, "predictive") vs DINOv2 (self-distillation with a global objective), 3 sizes each. Readouts: `cls` (E2 / Huh) and `mean` over patch tokens.
- **Measurements** per (language readout × vision readout), for 2 preprocessings (`huh` = 95th-percentile clip + L2, the Aristotelian setting; `noclip` = L2 only):
  - (a) the Aristotelian layer search: raw max CKA, calibrated CKA, held-out unbiased CKA, mutual k-NN;
  - (b) the final layer only (the model's own output): calibrated CKA of that fixed pair, unbiased CKA, mutual k-NN.

**Pre-registered predictions** (written before any E6 result):
- **P1 (Mayug):** at each size, Qwen3-Embedding (`last`, its native readout) > Qwen3-Base (`mean`, the E2 readout) on calibrated CKA and held-out unbiased CKA, against DINOv2 `cls`. Positive in ≥ 8 of 9 (size × DINOv2) cells.
- **P2 (convergence where it should be):** the slope over log2(parameters) of held-out unbiased CKA is larger for the embedding family than for the decoder family.
- **P3 (global organisation vs "like with like"):** the encoder − decoder gap is ≥ 0 with MAE as well. "Like with like" predicts it to be < 0 with MAE.
- **P4 (pooling):** for decoders, `prompt` or `last` ≥ `mean` on the final-layer measurements. Exploratory; no direction pre-registered for MAE `cls` vs `mean`.

**How to run.** `SMOKE_TEST = True` first (~10 min: Qwen3-0.6B-Base, Qwen3-Embedding-0.6B, mpnet × MAE-B, DINOv2-B, 50 permutations). It ends with a **CHECKER** table and `e6_smoke_results.zip`. Then set `SMOKE_TEST = False` and *Run all* (~1.5–2 h on a G4/H100). It ends with `e6_results.zip`. Everything is saved to Drive and resumes after a disconnect.

In [ ]:
# ============================== CONFIG ==============================
import os
SMOKE_TEST = False      # smoke and first full run done (12 Oct); this version only re-aligns (features are on Drive)
N_PERMS = 200                # Aristotelian paper's PRH runs: 200
ALPHA = 0.05
TOPK = 10
VARIANTS_E6 = ("huh", "noclip")
B_BOOT = 2000
SAVE_TO_DRIVE = True
SEED = 0

LANG_FULL = [   # (model, kind, readouts)
    ("Qwen/Qwen3-0.6B-Base", "decoder", ("mean", "last", "prompt")),
    ("Qwen/Qwen3-4B-Base", "decoder", ("mean", "last", "prompt")),
    ("Qwen/Qwen3-8B-Base", "decoder", ("mean", "last", "prompt")),
    ("Qwen/Qwen3-Embedding-0.6B", "embedding", ("last", "mean")),
    ("Qwen/Qwen3-Embedding-4B", "embedding", ("last", "mean")),
    ("Qwen/Qwen3-Embedding-8B", "embedding", ("last", "mean")),
    ("sentence-transformers/all-mpnet-base-v2", "encoder", ("mean", "cls")),
]
VIS_FULL = ["vit_base_patch16_224.mae", "vit_large_patch16_224.mae", "vit_huge_patch14_224.mae",
            "vit_base_patch14_dinov2.lvd142m", "vit_large_patch14_dinov2.lvd142m", "vit_giant_patch14_dinov2.lvd142m"]
NATIVE = {"decoder": "mean", "embedding": "last", "encoder": "mean"}   # E2 readout for decoders; own pooling for the others
if SMOKE_TEST:
    LANG = [LANG_FULL[0], LANG_FULL[3], LANG_FULL[6]]
    VIS = [VIS_FULL[0], VIS_FULL[3]]
    N_PERMS, B_BOOT = 50, 200
else:
    LANG, VIS = LANG_FULL, VIS_FULL

In [ ]:
# ============================== SETUP ==============================
import sys, subprocess, json, time, glob, shutil, math
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "transformers", "timm", "accelerate", "psutil", "huggingface_hub",
                    "datasets", "sentencepiece", "protobuf", "sentence-transformers"], check=False)
    if SAVE_TO_DRIVE:
        from google.colab import drive; drive.mount("/content/drive")
    try:
        from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        pass
ROOT = os.environ.get("PRH_ROOT") or ("/content/drive/MyDrive/prh_audit" if (IN_COLAB and SAVE_TO_DRIVE) else "/content/prh_audit")
RUN = os.path.join(ROOT, "e6_smoke" if SMOKE_TEST else "e6")
FEAT_DIR, RES_DIR, FIG_DIR = (os.path.join(RUN, d) for d in ("features", "results", "figs"))
for d in (FEAT_DIR, RES_DIR, FIG_DIR): os.makedirs(d, exist_ok=True)
import torch, numpy as np, pandas as pd
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
DEV = "cuda" if torch.cuda.is_available() else "cpu"
LOG = os.path.join(RUN, "log.txt")
def log(*a):
    s = time.strftime("%H:%M:%S ") + " ".join(str(x) for x in a); print(s); open(LOG, "a").write(s + "\n")
import transformers, timm
log("device:", torch.cuda.get_device_name(0) if DEV == "cuda" else "cpu", "| torch", torch.__version__,
    "| transformers", transformers.__version__, "| timm", timm.__version__, "| run dir:", RUN)

In [ ]:
# ============================== ENGINE (identical to the E2 audit, v3) ==============================
# ===================== Alignment engine (all layer pairs at once, GPU) =====================
# Every metric is computed for ALL (layer_x, layer_y) pairs with one matmul on flattened
# n x n matrices. Permutation nulls permute one side's matrices with the SAME permutation
# for every layer, so the max-over-layer-pairs aggregate is calibrated as in the Aristotelian paper.
import math, numpy as np, torch

VARIANTS = ("huh", "noclip", "std")
# huh    : clamp |x| at the per-sample 95th percentile (averaged), then L2-normalise each sample
#          (exactly Huh et al.'s prepare_features + compute_score(normalize=True))
# noclip : L2-normalise only
# std    : per-layer, per-dimension z-score over samples, then L2-normalise

def huh_clip(feats, q=0.95):
    """Huh et al. remove_outliers(exact=False): per-sample q-quantile of |x| over all layers & dims
    (linear interpolation, as torch.quantile), averaged over samples, then clamp. Uses kthvalue so it
    also works on tensors larger than torch.quantile's size limit."""
    a = feats.abs().flatten(start_dim=1); m = a.shape[1]
    pos = q * (m - 1); lo = int(math.floor(pos)); hi = int(math.ceil(pos)); w = pos - lo
    qs = []
    for i in range(a.shape[0]):
        v_lo = a[i].kthvalue(lo + 1).values
        v_hi = a[i].kthvalue(hi + 1).values if hi != lo else v_lo
        qs.append(v_lo + (v_hi - v_lo) * w)
    t = torch.stack(qs).mean()
    return feats.clamp(-t, t)

def preprocess(feats, variant):
    """feats: N x L x D float32 tensor -> list of L tensors (N x D), ready for metrics."""
    if variant == "huh":
        f = huh_clip(feats)
    elif variant == "noclip":
        f = feats
    elif variant == "std":
        f = (feats - feats.mean(0, keepdim=True)) / (feats.std(0, keepdim=True) + 1e-6)
    else:
        raise ValueError(variant)
    return [torch.nn.functional.normalize(f[:, l, :], dim=-1) for l in range(f.shape[1])]

def _center(K):
    m = K.mean(0)
    return K - m[None, :] - m[:, None] + m.mean()

class SideStats:
    """Per-model, per-variant matrices needed for every metric, flattened over layers."""
    def __init__(self, layers, k=10, half_idx=None):
        n = layers[0].shape[0]; self.n = n; self.L = len(layers); self.k = k
        Kc, Kt, adj, reff = [], [], [], []
        Kt_a, Kt_b, Kc_a, Kc_b = [], [], [], []
        ia, ib = half_idx if half_idx is not None else (None, None)
        for X in layers:
            K = X @ X.T
            c = _center(K); Kc.append(c.flatten())
            Kd = X.double() @ X.double().T                                     # fp64: unbiased HSIC cancels large terms
            t = Kd.clone(); t.fill_diagonal_(0); Kt.append(t.flatten())
            reff.append((torch.trace(c) ** 2 / (c * c).sum()).item())
            S = K.clone(); S.fill_diagonal_(-1e8)
            idx = S.topk(k, dim=1).indices
            A = torch.zeros(n, n, device=X.device); A.scatter_(1, idx, 1.0); adj.append(A.flatten())
            if ia is not None:
                for sub, Kt_s, Kc_s in ((ia, Kt_a, Kc_a), (ib, Kt_b, Kc_b)):
                    Ks = K[sub][:, sub]; Kc_s.append(_center(Ks).flatten())
                    ts = Kd[sub][:, sub].clone(); ts.fill_diagonal_(0); Kt_s.append(ts.flatten())
        self.Kc = torch.stack(Kc); self.Kc_norm = self.Kc.norm(dim=1)
        self.Kt = torch.stack(Kt); self.adj = torch.stack(adj)
        self.reff = np.array(reff)
        self.hu = self._hsic_parts(self.Kt, n)
        if ia is not None:
            self.Kc_a = torch.stack(Kc_a); self.Kc_b = torch.stack(Kc_b)
            self.Kc_a_norm = self.Kc_a.norm(dim=1); self.Kc_b_norm = self.Kc_b.norm(dim=1)
            self.hu_a = self._hsic_parts(torch.stack(Kt_a), len(ia))
            self.hu_b = self._hsic_parts(torch.stack(Kt_b), len(ib))

    @staticmethod
    def _hsic_parts(Kt, n):
        M = Kt.view(Kt.shape[0], n, n)
        return dict(flat=Kt, tot=Kt.sum(1), rows=M.sum(2), n=n)

def _hsic_u_pairs(A, B):
    """Unbiased HSIC (Song et al. 2012) for all layer pairs. A, B: dicts from _hsic_parts."""
    n = A["n"]
    t1 = A["flat"] @ B["flat"].T
    t2 = A["tot"][:, None] * B["tot"][None, :] / ((n - 1) * (n - 2))
    t3 = 2.0 * (A["rows"] @ B["rows"].T) / (n - 2)
    return (t1 + t2 - t3) / (n * (n - 3))

def _hsic_u_self(A):
    n = A["n"]
    return ((A["flat"] ** 2).sum(1) + A["tot"] ** 2 / ((n - 1) * (n - 2))
            - 2.0 * (A["rows"] ** 2).sum(1) / (n - 2)) / (n * (n - 3))

def unbiased_cka_pairs(A, B, rel_tol=1e-6):
    """fp64. Layers whose self-HSIC is ~0 or negative (near-constant representations, e.g. a ViT's
    first-block CLS token) are undefined for CKA and returned as NaN instead of dividing by ~0."""
    sa, sb = _hsic_u_self(A), _hsic_u_self(B)
    va = sa > rel_tol * sa.max().clamp_min(1e-300); vb = sb > rel_tol * sb.max().clamp_min(1e-300)
    u = _hsic_u_pairs(A, B) / torch.sqrt(sa.clamp_min(1e-300)[:, None] * sb.clamp_min(1e-300)[None, :])
    u[~va, :] = float("nan"); u[:, ~vb] = float("nan")
    return u.float()

def _nanargmax2(M):
    i = int(torch.nan_to_num(M, nan=-float("inf")).argmax()); return divmod(i, M.shape[1])

def linear_cka_pairs(x, y):
    return (x.Kc @ y.Kc.T) / (x.Kc_norm[:, None] * y.Kc_norm[None, :])

def mknn_pairs(x, y):
    return (x.adj @ y.adj.T) / (x.n * x.k)

def _perm_flat_index(p, n):
    return (p[:, None] * n + p[None, :]).flatten()

def calibrate(obs, null, alpha=0.05, smax=1.0):
    K = len(null)
    allv = np.sort(np.append(null, obs))
    tau = allv[int(math.ceil((1 - alpha) * (K + 1))) - 1]
    cal = max((obs - tau) / (smax - tau), 0.0) if smax > tau else 0.0
    p = (1 + np.sum(null >= obs)) / (K + 1)
    return cal, tau, p

def align_pair(x, y, perms, alpha=0.05, perms_half=None):
    """x: language SideStats, y: vision SideStats. Returns dict of scalar results + matrices."""
    n = x.n
    cka = linear_cka_pairs(x, y); ucka = unbiased_cka_pairs(x.hu, y.hu); mk = mknn_pairs(x, y)
    i_c, j_c = divmod(int(cka.argmax()), cka.shape[1])
    i_u, j_u = _nanargmax2(ucka)
    i_m, j_m = divmod(int(mk.argmax()), mk.shape[1])
    null_cka_max, null_mk_max, null_cka_fixed, null_arg = [], [], [], []
    for p in perms:
        fi = _perm_flat_index(p, n)
        Kc_p = y.Kc.index_select(1, fi); A_p = y.adj.index_select(1, fi)
        s = (x.Kc @ Kc_p.T) / (x.Kc_norm[:, None] * y.Kc_norm[None, :])
        a_ = int(s.argmax()); null_arg.append(a_)
        null_cka_max.append(s.view(-1)[a_].item()); null_cka_fixed.append(s[i_c, j_c].item())
        null_mk_max.append(((x.adj @ A_p.T) / (n * x.k)).max().item())
    cka_max = cka.max().item(); mk_max = mk.max().item()
    cal_agg, tau_agg, p_agg = calibrate(cka_max, np.array(null_cka_max), alpha)
    cal_fix, tau_fix, _ = calibrate(cka_max, np.array(null_cka_fixed), alpha)
    mk_cal, mk_tau, mk_p = calibrate(mk_max, np.array(null_mk_max), alpha)
    out = dict(cka_max=cka_max, cka_cal=cal_agg, cka_tau=tau_agg, cka_p=p_agg,
               cka_cal_fixedpair=cal_fix, cka_tau_fixedpair=tau_fix,
               cka_null_mean=float(np.mean(null_cka_max)),
               ucka_max=float(np.nanmax(ucka.cpu().numpy())), mknn_max=mk_max, mknn_cal=mk_cal, mknn_tau=mk_tau, mknn_p=mk_p,
               best_cka=(i_c, j_c), best_ucka=(i_u, j_u), best_mknn=(i_m, j_m),
               reff_x=float(x.reff[i_c]), reff_y=float(y.reff[j_c]))
    out["pred_shrink"] = 1.0 / math.sqrt((1 + out["reff_x"] / n) * (1 + out["reff_y"] / n))
    # which layer pair sets the aggregate null threshold most often, and how low-rank is it?
    mode = max(set(null_arg), key=null_arg.count); im, jm = divmod(mode, cka.shape[1])
    out.update(nullmax_pair=(im, jm), nullmax_pair_freq=null_arg.count(mode) / len(null_arg),
               nullmax_reff_x=float(x.reff[im]), nullmax_reff_y=float(y.reff[jm]))
    if hasattr(x, "hu_a") and hasattr(y, "hu_a"):
        ua = unbiased_cka_pairs(x.hu_a, y.hu_a)
        ia, ja = _nanargmax2(ua)
        ub = unbiased_cka_pairs(x.hu_b, y.hu_b)
        out["ucka_heldout"] = ub[ia, ja].item(); out["best_heldout"] = (ia, ja)
        # held-out CALIBRATED CKA: pick the layer pair on half A by UNBIASED CKA (raw CKA is inflated by the
        # chance offset and would favour high-rank layers), then calibrate that single pair's raw CKA on half B
        ic, jc = ia, ja
        xb, yb = x.Kc_b[ic], y.Kc_b[jc]; nb = int(round(math.sqrt(xb.numel())))
        obs = (xb @ yb / (x.Kc_b_norm[ic] * y.Kc_b_norm[jc])).item()
        nulls = [(xb @ yb.index_select(0, _perm_flat_index(p, nb)) / (x.Kc_b_norm[ic] * y.Kc_b_norm[jc])).item()
                 for p in (perms_half or [])]
        if nulls:
            c_, t_, p_ = calibrate(obs, np.array(nulls), alpha)
            out.update(cka_heldout_raw=obs, cka_heldout_cal=c_, cka_heldout_tau=t_, best_heldout_cka=(ic, jc))
    mats = dict(cka=cka.cpu().numpy(), ucka=ucka.cpu().numpy(), mknn=mk.cpu().numpy())
    return out, mats

def layer_diagnostics(feats):
    """Outlier diagnostics on raw (un-normalised) features. feats: N x L x D."""
    rows = []
    for l in range(feats.shape[1]):
        f = feats[:, l, :]
        v = f.var(0); vs = torch.sort(v, descending=True).values; tot = v.sum()
        absf = f.abs()
        fc = f - f.mean(0, keepdim=True)
        G = fc @ fc.T
        rows.append(dict(layer=l, top1_var_share=(vs[0] / tot).item(), top10_var_share=(vs[:10].sum() / tot).item(),
                         max_over_median_abs=(absf.max() / absf.median().clamp_min(1e-12)).item(),
                         reff_raw=(torch.trace(G) ** 2 / (G * G).sum()).item()))
    return rows

In [ ]:
# ============================== E6 EXTRACTION CODE ==============================
# ===================== E6 feature extraction: several readouts per model, every layer =====================
# Language: hidden states of every layer (embeddings + each block; the last entry is after the final norm, i.e. the
#           model's own output), read out per caption as
#             mean   = mean over non-pad tokens (Huh et al. / E2 protocol)
#             last   = last token (left padding, so position -1 is always the caption's last real token)
#             prompt = last token of a PromptEOL template (Jiang et al. 2023), a training-free decoder->encoder readout
#             cls    = first token (encoders only)
# Vision:   output of every transformer block, read out as cls (token 0, = E2 / Huh et al.) and mean over patch tokens
#           (prefix tokens excluded), timm transforms.
import gc, math, os, shutil
import torch, transformers

DEV = "cuda" if torch.cuda.is_available() else "cpu"
PROMPT_EOL = 'This sentence : "{}" means in one word:"'

def _dtype_kw(dtype):
    major, minor = (int(x) for x in transformers.__version__.split(".")[:2])
    return {"dtype": dtype} if (major, minor) >= (4, 56) else {"torch_dtype": dtype}

def hf_cache_clear(name):
    from huggingface_hub import constants
    shutil.rmtree(os.path.join(constants.HF_HUB_CACHE, "models--" + name.replace("/", "--")), ignore_errors=True)
    shutil.rmtree(getattr(constants, "HF_XET_CACHE", os.path.join(constants.HF_HOME, "xet")), ignore_errors=True)

@torch.no_grad()
def extract_text(name, texts, kind, readouts, batch_size=32, act_budget_gb=6.0):
    """kind: 'decoder' | 'embedding' (decoder-architecture embedding model, last-token pooling) | 'encoder' (BERT-like).
    Returns {readout: dict(feats=N x L x D bf16, ...)} from the SAME weights."""
    from transformers import AutoModel, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(name)
    left = kind in ("decoder", "embedding")
    if left:
        tok.padding_side = "left"
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
    dtype = torch.bfloat16 if left else torch.float32            # encoders (mpnet) run in fp32, as sentence-transformers does
    model = AutoModel.from_pretrained(name, **_dtype_kw(dtype)).to(DEV).eval()
    n_params = sum(p.numel() for p in model.parameters())
    c = model.config.get_text_config() if hasattr(model.config, "get_text_config") else model.config
    L, D = c.num_hidden_layers, c.hidden_size
    out = {}
    groups = {"prompt": [r for r in readouts if r == "prompt"], "plain": [r for r in readouts if r != "prompt"]}
    for grp, rs in groups.items():
        if not rs: continue
        inp = [PROMPT_EOL.format(t) for t in texts] if grp == "prompt" else list(texts)
        lens = [len(x) for x in tok(inp, add_special_tokens=True)["input_ids"]]
        T = max(lens)
        per_seq = T * (L + 1) * D * 4 * 2                       # all hidden states, fp32 pooling copies
        bs = max(1, min(batch_size, int(act_budget_gb * 1e9 // per_seq)))
        acc = {r: [] for r in rs}
        for i in range(0, len(inp), bs):
            b = tok(inp[i:i + bs], padding="longest", return_tensors="pt").to(DEV)
            am = b["attention_mask"]
            if left:
                assert bool(am[:, -1].all()), "left padding expected: last position must be a real token"
            hs = model(**b, output_hidden_states=True).hidden_states   # tuple of L+1 tensors B x T x D
            w = am.unsqueeze(-1).float()
            for r in rs:
                if r == "mean":
                    v = [((h.float() * w).sum(1) / w.sum(1)) for h in hs]
                elif r in ("last", "prompt"):
                    v = [h[:, -1].float() for h in hs]
                elif r == "cls":
                    v = [h[:, 0].float() for h in hs]
                else:
                    raise ValueError(r)
                acc[r].append(torch.stack(v, 1).cpu().to(torch.bfloat16))   # B x (L+1) x D
            del hs
        for r in rs:
            out[r] = dict(feats=torch.cat(acc[r]), num_params=int(n_params), kind=kind, readout=r,
                          dtype=str(dtype), batch_size=bs, max_tokens=T)
    # Self-check for the checker: the last hidden_states entry equals the model's own output (last_hidden_state)
    b = tok(list(texts[:8]), padding="longest", return_tensors="pt").to(DEV)
    o = model(**b, output_hidden_states=True)
    same = bool(torch.equal(o.hidden_states[-1], o.last_hidden_state))
    for r in out: out[r]["final_is_model_output"] = same
    del model; gc.collect(); torch.cuda.empty_cache()
    return out

@torch.no_grad()
def extract_vision2(name, images, batch_size=64):
    """Returns {'cls': ..., 'mean': ...}, each N x L x D bf16 (one entry per transformer block)."""
    import timm
    from timm.data import resolve_data_config
    from timm.data.transforms_factory import create_transform
    model = timm.create_model(name, pretrained=True).eval().to(DEV)
    n_params = sum(p.numel() for p in model.parameters())
    tf = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))
    npre = int(getattr(model, "num_prefix_tokens", 1))
    store = {}
    def hook(k):
        def f(m, i, o):
            store[k] = (o[:, 0, :].float().cpu(), o[:, npre:, :].float().mean(1).cpu())
        return f
    hooks = [blk.register_forward_hook(hook(k)) for k, blk in enumerate(model.blocks)]
    cls, mean = [], []
    for i in range(0, len(images), batch_size):
        x = torch.stack([tf(im.convert("RGB")) for im in images[i:i + batch_size]]).to(DEV)
        store.clear(); model(x)
        cls.append(torch.stack([store[k][0] for k in range(len(model.blocks))], 1))
        mean.append(torch.stack([store[k][1] for k in range(len(model.blocks))], 1))
    for h in hooks: h.remove()
    del model; gc.collect(); torch.cuda.empty_cache()
    base = dict(num_params=int(n_params), num_prefix_tokens=npre)
    return {"cls": dict(feats=torch.cat(cls).to(torch.bfloat16), readout="cls", **base),
            "mean": dict(feats=torch.cat(mean).to(torch.bfloat16), readout="mean", **base)}

In [ ]:
# ============================== DATA: WIT-1024 (Huh et al.), identical to E2 ==============================
from datasets import load_dataset
ds = load_dataset("minhuh/prh", revision="wit_1024", split="train")
texts = [str(x["text"][0]) for x in ds]; images = [x["image"] for x in ds]
N = len(texts); assert N == 1024, N
log(f"dataset: {N} pairs; first caption: {texts[0][:80]!r}")
def fpath(model, readout):
    return os.path.join(FEAT_DIR, f"{model.replace('/', '_')}__{readout}.pt")

In [ ]:
# ============================== EXTRACT ==============================
for name in VIS:
    if all(os.path.exists(fpath(name, r)) for r in ("cls", "mean")):
        log("vision exists:", name); continue
    t0 = time.time()
    try:
        out = extract_vision2(name, images)
        for r, d in out.items(): torch.save(d, fpath(name, r))
        log(f"vision done: {name} {tuple(out['cls']['feats'].shape)} ({time.time()-t0:.0f}s)")
    except Exception as e:
        log("VISION FAILED:", name, repr(e)[:300])
    finally:
        hf_cache_clear("timm/" + name)
for name, kind, rs in LANG:
    if all(os.path.exists(fpath(name, r)) for r in rs):
        log("language exists:", name); continue
    t0 = time.time()
    try:
        out = extract_text(name, texts, kind, rs)
        for r, d in out.items(): torch.save(d, fpath(name, r))
        log(f"language done: {name} [{kind}] readouts={list(out)} {tuple(next(iter(out.values()))['feats'].shape)} "
            f"batch={next(iter(out.values()))['batch_size']} ({time.time()-t0:.0f}s)")
    except Exception as e:
        log("LANGUAGE FAILED:", name, repr(e)[:300])
    finally:
        hf_cache_clear(name)

In [ ]:
# ============================== ALIGN: every language readout x every vision readout ==============================
g = torch.Generator().manual_seed(SEED)
PERMS = [torch.randperm(N, generator=g).to(DEV) for _ in range(N_PERMS)]
_split = torch.randperm(N, generator=g)
HALF = (_split[: N // 2].to(DEV), _split[N // 2:].to(DEV))
PERMS_HALF = [torch.randperm(N - N // 2, generator=g).to(DEV) for _ in range(N_PERMS)]
L_SETS = [(m, k, r) for m, k, rs in LANG for r in rs if os.path.exists(fpath(m, r))]
V_SETS = [(m, r) for m in VIS for r in ("cls", "mean") if os.path.exists(fpath(m, r))]
log(f"aligning {len(L_SETS)} language feature sets x {len(V_SETS)} vision feature sets x {len(VARIANTS_E6)} preprocessings")

DROPPED = {}
def stats(path, variant, final_only):
    f = torch.load(path)["feats"].to(DEV).float()
    if final_only: f = f[:, -1:, :]
    layers = preprocess(f, variant)
    # A layer whose readout is the same vector for every caption (e.g. layer 0 of a last-token / prompt / [CLS] readout:
    # the input embedding of one fixed token) has zero variance; its CKA is 0/0. Drop such layers from the layer search.
    keep = [l for l, X in enumerate(layers) if float((X - X.mean(0)).pow(2).sum()) > 1e-6 * X.shape[0]]
    if len(keep) < len(layers): DROPPED[(os.path.basename(path), variant)] = [l for l in range(len(layers)) if l not in keep]
    s = SideStats([layers[l] for l in keep], k=TOPK, half_idx=HALF); s.kept_layers = keep; del f, layers; return s

for variant in VARIANTS_E6:
    for final_only in (False, True):
        tag = f"{variant}_{'final' if final_only else 'layers'}"
        out_csv = os.path.join(RES_DIR, f"align2_{tag}.csv")   # v2: constant layers dropped
        done = pd.read_csv(out_csv) if os.path.exists(out_csv) else pd.DataFrame(columns=["lang", "lang_readout"])
        todo = [x for x in L_SETS if not ((done.lang == x[0]) & (done.lang_readout == x[2])).any()]
        if not todo:
            log("done:", tag); continue
        vs = {(m, r): stats(fpath(m, r), variant, final_only) for m, r in V_SETS}
        for m, k, r in todo:
            t1 = time.time(); xs = stats(fpath(m, r), variant, final_only); rows = []
            for (vm, vr), ys in vs.items():
                o, _ = align_pair(xs, ys, PERMS, alpha=ALPHA, perms_half=PERMS_HALF)
                rows.append(dict(variant=variant, scope="final" if final_only else "layers", lang=m, lang_kind=k, lang_readout=r,
                                 vision=vm, vision_readout=vr, **{a: (str(b) if isinstance(b, tuple) else b) for a, b in o.items()}))
            done = pd.concat([done, pd.DataFrame(rows)], ignore_index=True); done.to_csv(out_csv, index=False)
            log(f"[{tag}] {m} / {r}: {len(rows)} pairs ({time.time()-t1:.0f}s)")
            del xs; torch.cuda.empty_cache()
        del vs; torch.cuda.empty_cache()
log("constant layers dropped from the layer search:", {f"{k[0]}|{k[1]}": v for k, v in DROPPED.items()})

In [ ]:
# ============================== CHECKER (read the PASS/FAIL table) ==============================
checks = []
def check(name, ok, detail=""):
    checks.append(dict(check=name, result="PASS" if ok else "FAIL", detail=str(detail)[:200]))
wl = open(LOG).read()
check("no extraction failures", "FAILED" not in wl, [l for l in wl.splitlines() if "FAILED" in l][:3])
exp_sets = [(m, r) for m, k, rs in LANG for r in rs] + [(m, r) for m in VIS for r in ("cls", "mean")]
missing = [f"{m}/{r}" for m, r in exp_sets if not os.path.exists(fpath(m, r))]
check("all feature files present", not missing, missing[:5])
bad = []
for m, r in exp_sets:
    if os.path.exists(fpath(m, r)):
        d = torch.load(fpath(m, r), mmap=True); f = d["feats"]
        if f.shape[0] != N or not torch.isfinite(f.float()).all(): bad.append(f"{m}/{r}")
check("features: N = 1024 rows and all finite", not bad, bad[:5])
fo = [torch.load(fpath(m, r), mmap=True).get("final_is_model_output", True) for m, k, rs in LANG for r in rs if os.path.exists(fpath(m, r))]
check("last layer read out = the model's own output (last_hidden_state)", all(fo), fo)
# readouts must differ from each other (catches a pooling bug that returns the same tensor twice)
for m, k, rs in LANG:
    if len(rs) > 1 and all(os.path.exists(fpath(m, r)) for r in rs):
        a, b = (torch.load(fpath(m, r))["feats"][:, -1].float() for r in rs[:2])
        check(f"readouts differ ({m.split('/')[-1]}: {rs[0]} vs {rs[1]})", not torch.allclose(a, b), float((a - b).abs().max()))
# mpnet mean readout reproduces sentence-transformers' own embedding
try:
    from sentence_transformers import SentenceTransformer
    st_name = "sentence-transformers/all-mpnet-base-v2"
    if os.path.exists(fpath(st_name, "mean")):
        st = SentenceTransformer(st_name, device=DEV)
        e_st = torch.tensor(st.encode(texts[:64], normalize_embeddings=True))
        ours = torch.nn.functional.normalize(torch.load(fpath(st_name, "mean"))["feats"][:64, -1].float(), dim=-1)
        cos = float((e_st * ours).sum(-1).min())
        check("mpnet mean readout = sentence-transformers embedding (min cosine > 0.999)", cos > 0.999, round(cos, 6))
        del st; hf_cache_clear(st_name)
except Exception as e:
    check("mpnet vs sentence-transformers (could not run)", False, repr(e)[:150])
res_all = pd.concat([pd.read_csv(f) for f in glob.glob(os.path.join(RES_DIR, "align2_*.csv"))], ignore_index=True)
check("alignment rows complete", len(res_all) == len(L_SETS) * len(V_SETS) * len(VARIANTS_E6) * 2,
      f"{len(res_all)} vs {len(L_SETS) * len(V_SETS) * len(VARIANTS_E6) * 2}")
check("calibrated CKA in [0, 1] and <= raw", bool(((res_all.cka_cal >= 0) & (res_all.cka_cal <= res_all.cka_max + 1e-9)).all()))
lay = res_all[res_all.scope == "layers"]
tau_pred = np.sqrt(lay.nullmax_reff_x * lay.nullmax_reff_y) / N
check("threshold = chance level of the highest-rank layer pair (r > 0.99, as in E2)", float(np.corrcoef(tau_pred, lay.cka_tau)[0, 1]) > 0.99,
      round(float(np.corrcoef(tau_pred, lay.cka_tau)[0, 1]), 5))
# null: a shuffled pairing must give calibrated CKA = 0
m0, k0, r0 = L_SETS[0]; v0, vr0 = V_SETS[0]
xs = stats(fpath(m0, r0), "huh", True); zero = 0
for s_ in range(10):
    p = torch.randperm(N, generator=torch.Generator().manual_seed(100 + s_)).to(DEV)
    f = torch.load(fpath(v0, vr0))["feats"].to(DEV).float()[:, -1:, :][p]
    ys = SideStats(preprocess(f, "huh"), k=TOPK, half_idx=HALF)
    o, _ = align_pair(xs, ys, PERMS, alpha=ALPHA); zero += int(o["cka_cal"] == 0)
check("null: shuffled pairs give calibrated CKA = 0 (>= 9/10)", zero >= 9, f"{zero}/10")
checks = pd.DataFrame(checks); checks.to_csv(os.path.join(RUN, "checker.csv"), index=False)
print(checks.to_string(index=False)); log("checker:", (checks.result == "PASS").sum(), "PASS /", len(checks))
if SMOKE_TEST:
    import zipfile
    zp = os.path.join(RUN, "e6_smoke_results.zip")
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
        for f in [LOG, os.path.join(RUN, "checker.csv")] + glob.glob(os.path.join(RES_DIR, "*.csv")):
            z.write(f, os.path.relpath(f, RUN))
    if IN_COLAB:
        from google.colab import files; files.download(zp)
    raise SystemExit("Smoke test finished: read the CHECKER table, send e6_smoke_results.zip, then set SMOKE_TEST = False and Run all.")

In [ ]:
# ============================== PRE-REGISTERED ANALYSIS ==============================
import matplotlib; import matplotlib.pyplot as plt
res = res_all.copy()
psize = {m: torch.load(fpath(m, rs[0]), mmap=True)["num_params"] for m, k, rs in LANG if os.path.exists(fpath(m, rs[0]))}
psize.update({m: torch.load(fpath(m, "cls"), mmap=True)["num_params"] for m in VIS if os.path.exists(fpath(m, "cls"))})
res["lang_params"] = res.lang.map(psize); res["vision_params"] = res.vision.map(psize)
res["size"] = res.lang.str.extract(r"(0\.6B|4B|8B)")[0]
res["vfam"] = np.where(res.vision.str.contains("mae"), "MAE", "DINOv2")
res.to_csv(os.path.join(RES_DIR, "e6_all.csv"), index=False)
PRIMARY = ["cka_cal", "ucka_heldout"]                  # pre-registered (layer search, huh)
ALLM = ["cka_max", "cka_cal", "ucka_heldout", "ucka_max", "mknn_max", "mknn_cal"]
rng = np.random.default_rng(SEED)
def boot_ci(df, value, cluster, B=B_BOOT):
    """Mean of `value` and a 95% CI resampling clusters (vision models)."""
    cl = df[cluster].unique(); est = df[value].mean(); bs = []
    for _ in range(B):
        pick = rng.choice(cl, len(cl)); bs.append(pd.concat([df[df[cluster] == c] for c in pick])[value].mean())
    return est, float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))

def sel(scope="layers", variant="huh", kind=None, readout=None, vfam=None, vreadout=None):
    d = res[(res.scope == scope) & (res.variant == variant)]
    if kind: d = d[d.lang_kind == kind]
    if readout: d = d[d.lang_readout == readout]
    if vfam: d = d[d.vfam == vfam]
    if vreadout: d = d[d.vision_readout == vreadout]
    return d

rows = []
# P1: embedding(last) - decoder(mean), same size, DINOv2 cls
for scope in ("layers", "final"):
    for variant in VARIANTS_E6:
        for met in ALLM if scope == "layers" else ["cka_max", "cka_cal", "ucka_max", "mknn_max"]:
            e = sel(scope, variant, "embedding", "last", "DINOv2", "cls")[["size", "vision", met]]
            dcd = sel(scope, variant, "decoder", "mean", "DINOv2", "cls")[["size", "vision", met]]
            j = e.merge(dcd, on=["size", "vision"], suffixes=("_enc", "_dec")); j["delta"] = j[f"{met}_enc"] - j[f"{met}_dec"]
            if len(j):
                est, lo, hi = boot_ci(j, "delta", "vision")
                rows.append(dict(test="P1 Qwen3-Embedding(last) - Qwen3-Base(mean), DINOv2 cls", scope=scope, variant=variant, metric=met,
                                 delta=est, ci_low=lo, ci_high=hi, positive=f"{int((j.delta > 0).sum())}/{len(j)}",
                                 primary=(scope == "layers" and variant == "huh" and met in PRIMARY)))
# P3: encoder - decoder gap by vision family (2x2), each vision readout
for variant in VARIANTS_E6:
    for vr in ("cls", "mean"):
        for vf in ("DINOv2", "MAE"):
            for met in PRIMARY + ["mknn_max"]:
                e = sel("layers", variant, "embedding", "last", vf, vr)[["size", "vision", met]]
                dcd = sel("layers", variant, "decoder", "mean", vf, vr)[["size", "vision", met]]
                j = e.merge(dcd, on=["size", "vision"], suffixes=("_enc", "_dec")); j["delta"] = j[f"{met}_enc"] - j[f"{met}_dec"]
                if len(j):
                    est, lo, hi = boot_ci(j, "delta", "vision")
                    rows.append(dict(test=f"P3 encoder - decoder gap | {vf} {vr}", scope="layers", variant=variant, metric=met,
                                     delta=est, ci_low=lo, ci_high=hi, positive=f"{int((j.delta > 0).sum())}/{len(j)}",
                                     primary=(variant == "huh" and vr == "cls" and met in PRIMARY)))
# P4: decoder readouts vs mean (final layer and layer search), DINOv2 cls
for scope in ("final", "layers"):
    for rd in ("last", "prompt"):
        for met in ["cka_cal", "ucka_max", "mknn_max"]:
            a = sel(scope, "huh", "decoder", rd, "DINOv2", "cls")[["size", "vision", met]]
            b = sel(scope, "huh", "decoder", "mean", "DINOv2", "cls")[["size", "vision", met]]
            j = a.merge(b, on=["size", "vision"], suffixes=("_a", "_b")); j["delta"] = j[f"{met}_a"] - j[f"{met}_b"]
            if len(j):
                est, lo, hi = boot_ci(j, "delta", "vision")
                rows.append(dict(test=f"P4 decoder {rd} - mean", scope=scope, variant="huh", metric=met, delta=est, ci_low=lo,
                                 ci_high=hi, positive=f"{int((j.delta > 0).sum())}/{len(j)}", primary=False))
tests = pd.DataFrame(rows); tests.to_csv(os.path.join(RES_DIR, "e6_tests.csv"), index=False)
print(tests.round(4).to_string(index=False))

# P2: scale slopes (held-out unbiased CKA and calibrated CKA vs log2 params), DINOv2 cls, per family
slopes = []
for kind, rd in (("embedding", "last"), ("decoder", "mean"), ("decoder", "prompt")):
    for met in PRIMARY + ["mknn_max"]:
        d = sel("layers", "huh", kind, rd, "DINOv2", "cls").copy(); d["lp"] = np.log2(d.lang_params.astype(float))
        if d.lang.nunique() < 2: continue
        s0 = np.polyfit(d.lp, d[met], 1)[0]; vits = d.vision.unique(); bs = []
        for _ in range(B_BOOT):
            bd = pd.concat([d[d.vision == v] for v in rng.choice(vits, len(vits))]); bs.append(np.polyfit(bd.lp, bd[met], 1)[0])
        slopes.append(dict(family=f"{kind}/{rd}", metric=met, slope_per_doubling=s0,
                           ci_low=float(np.percentile(bs, 2.5)), ci_high=float(np.percentile(bs, 97.5))))
slopes = pd.DataFrame(slopes); slopes.to_csv(os.path.join(RES_DIR, "e6_scale_slopes.csv"), index=False)
print(slopes.round(4).to_string(index=False))

# Summary table: mean over vision models, per language set x vision family/readout (layers, huh)
summ = (sel("layers", "huh").groupby(["lang", "lang_kind", "lang_readout", "vfam", "vision_readout"])[ALLM].mean().reset_index())
summ.to_csv(os.path.join(RES_DIR, "e6_summary_table.csv"), index=False)

# Figure: the 2x2 (held-out unbiased CKA), DINOv2 vs MAE x decoder vs embedding, by size
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8), sharey=True)
for ax, vf in zip(axs, ("DINOv2", "MAE")):
    for kind, rd, mk in (("embedding", "last", "o-"), ("decoder", "mean", "s--"), ("decoder", "prompt", "^:")):
        d = sel("layers", "huh", kind, rd, vf, "cls").groupby("lang").agg(p=("lang_params", "first"), y=("ucka_heldout", "mean")).sort_values("p")
        if len(d): ax.plot(d.p, d.y, mk, label=f"Qwen3 {kind} ({rd})")
    ax.set_xscale("log"); ax.set_title(f"vision: {vf} (cls)"); ax.set_xlabel("language model parameters"); ax.grid(alpha=.3)
axs[0].set_ylabel("held-out unbiased CKA (layer search, huh)"); axs[0].legend(fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(FIG_DIR, "fig_e6_2x2.png"), dpi=140); plt.show()

In [ ]:
# ============================== PACKAGE FOR CLAUDE ==============================
import zipfile
zp = os.path.join(RUN, "e6_results.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [LOG, os.path.join(RUN, "checker.csv")] + glob.glob(os.path.join(RES_DIR, "*.csv")) + glob.glob(os.path.join(FIG_DIR, "*.png")):
        if os.path.exists(f): z.write(f, os.path.relpath(f, RUN))
log("packaged", zp, f"{os.path.getsize(zp) / 1e6:.1f} MB")
if IN_COLAB:
    from google.colab import files; files.download(zp)